# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [3]:
# We use Type 1 as the primary Pokemon type to keep the two groups independent.
dragon_hp = df.loc[df["Type 1"] == "Dragon", "HP"]
grass_hp = df.loc[df["Type 1"] == "Grass", "HP"]

# H0: mean HP of Dragon Pokemon <= mean HP of Grass Pokemon
# H1: mean HP of Dragon Pokemon > mean HP of Grass Pokemon
alpha = 0.05
result = st.ttest_ind(dragon_hp, grass_hp, equal_var=False, alternative="greater")

print(f"Dragon sample size: {len(dragon_hp)}")
print(f"Grass sample size: {len(grass_hp)}")
print(f"Mean Dragon HP: {dragon_hp.mean():.2f}")
print(f"Mean Grass HP: {grass_hp.mean():.2f}")
print(f"t-statistic: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.6f}")

if result.pvalue < alpha:
    print("Reject H0: Dragon Pokemon have significantly higher HP than Grass Pokemon.")
else:
    print("Fail to reject H0: there is not enough evidence that Dragon Pokemon have higher HP.")


Dragon sample size: 32
Grass sample size: 70
Mean Dragon HP: 83.31
Mean Grass HP: 67.27
t-statistic: 3.3350
p-value: 0.000799
Reject H0: Dragon Pokemon have significantly higher HP than Grass Pokemon.


**Findings**

Using a one-sided Welch two-sample t-test at a 5% significance level, the p-value is approximately 0.0008. Since this is below 0.05, we reject the null hypothesis. Primary Dragon-type Pokemon have significantly higher average HP than primary Grass-type Pokemon in this dataset.


- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [4]:
stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
legendary = df[df["Legendary"]]
non_legendary = df[~df["Legendary"]]

# H0: Legendary and Non-Legendary Pokemon have the same mean for each stat.
# H1: Legendary and Non-Legendary Pokemon have different means for each stat.
alpha = 0.05
bonferroni_alpha = alpha / len(stats_cols)

results = []
for stat_col in stats_cols:
    test = st.ttest_ind(
        legendary[stat_col],
        non_legendary[stat_col],
        equal_var=False,
        alternative="two-sided"
    )
    results.append({
        "Stat": stat_col,
        "Legendary Mean": legendary[stat_col].mean(),
        "Non-Legendary Mean": non_legendary[stat_col].mean(),
        "t-statistic": test.statistic,
        "p-value": test.pvalue,
        "Significant at 5%": test.pvalue < alpha,
        "Significant after Bonferroni": test.pvalue < bonferroni_alpha,
    })

legendary_results = pd.DataFrame(results)
legendary_results


,Stat,Legendary Mean,Non-Legendary Mean,t-statistic,p-value,Significant at 5%,Significant after Bonferroni
0,HP,92.738462,67.182313,8.981370,1.002691e-13,True,True
1,Attack,116.676923,75.669388,10.438134,2.520372e-16,True,True
2,Defense,99.661538,71.559184,7.637078,4.826998e-11,True,True
3,Sp. Atk,122.184615,68.454422,13.417450,1.551461e-21,True,True
4,Sp. Def,105.938462,68.892517,10.015697,2.294933e-15,True,True
5,Speed,100.184615,65.455782,11.475044,1.049016e-18,True,True


**Findings**

Using two-sided Welch two-sample t-tests at a 5% significance level, all six stats have p-values far below 0.05. The differences also remain significant after a Bonferroni correction for the six tests. Legendary Pokemon have significantly different, and higher on average, HP, Attack, Defense, Sp. Atk, Sp. Def, and Speed than Non-Legendary Pokemon.


**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [6]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")

school_coordinates = (-118, 34)    # longitude, latitude
hospital_coordinates = (-122, 37)  # longitude, latitude


def euclidean_distance(longitude, latitude, coordinates):
    """Calculate Euclidean distance from a house/neighborhood to a fixed point."""
    point_longitude, point_latitude = coordinates
    return np.sqrt((longitude - point_longitude) ** 2 + (latitude - point_latitude) ** 2)


df["distance_to_school"] = euclidean_distance(
    df["longitude"],
    df["latitude"],
    school_coordinates
)
df["distance_to_hospital"] = euclidean_distance(
    df["longitude"],
    df["latitude"],
    hospital_coordinates
)

df["close_to_school_or_hospital"] = (
    (df["distance_to_school"] < 0.50) |
    (df["distance_to_hospital"] < 0.50)
)

df[[
    "longitude",
    "latitude",
    "median_house_value",
    "distance_to_school",
    "distance_to_hospital",
    "close_to_school_or_hospital"
]].head()


,longitude,latitude,median_house_value,distance_to_school,distance_to_hospital,close_to_school_or_hospital
0,-114.31,34.19,66900.0,3.694888,8.187319,False
1,-114.47,34.40,80100.0,3.552591,7.966235,False
2,-114.56,33.69,85700.0,3.453940,8.143077,False
3,-114.57,33.64,73400.0,3.448840,8.154416,False
4,-114.57,33.57,65500.0,3.456848,8.183508,False


In [7]:
close_prices = df.loc[df["close_to_school_or_hospital"], "median_house_value"]
far_prices = df.loc[~df["close_to_school_or_hospital"], "median_house_value"]

# H0: mean price of close houses <= mean price of far houses
# H1: mean price of close houses > mean price of far houses
alpha = 0.05
housing_result = st.ttest_ind(close_prices, far_prices, equal_var=False, alternative="greater")

print(f"Close sample size: {len(close_prices)}")
print(f"Far sample size: {len(far_prices)}")
print(f"Mean close house value: ${close_prices.mean():,.2f}")
print(f"Mean far house value: ${far_prices.mean():,.2f}")
print(f"t-statistic: {housing_result.statistic:.4f}")
print(f"p-value: {housing_result.pvalue:.6e}")

if housing_result.pvalue < alpha:
    print("Reject H0: houses close to a school or hospital are significantly more expensive.")
else:
    print("Fail to reject H0: there is not enough evidence that close houses are more expensive.")


Close sample size: 6829
Far sample size: 10171
Mean close house value: $246,951.98
Mean far house value: $180,678.44
t-statistic: 37.9923
p-value: 1.503248e-301
Reject H0: houses close to a school or hospital are significantly more expensive.


**Findings**

Using a one-sided Welch two-sample t-test at a 5% significance level, the p-value is effectively 0. The average value for houses close to a school or hospital is about $246,952, compared with about $180,678 for houses farther away. We reject the null hypothesis and conclude that houses close to a school or hospital are significantly more expensive in this dataset.
